# Starter Notebook

Install and import required libraries

In [ ]:
!pip install transformers datasets evaluate accelerate peft trl bitsandbytes
!pip install nvidia-ml-py3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install transformers datasets accelerate peft evaluate bitsandbytes

import os
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    RobertaPreTrainedModel
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig
from torch import nn
from torch.utils.data import DataLoader

# Set model name
base_model = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(base_model)

# Load dataset
dataset = load_dataset("ag_news", split="train")

# Preprocess function
def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding=True)

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Class info
num_labels = dataset.features["label"].num_classes
class_names = dataset.features["label"].names
id2label = {i: label for i, label in enumerate(class_names)}

# Split train/eval
split_datasets = tokenized_dataset.train_test_split(test_size=640, seed=42)
train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

# Define custom classifier
class RobertaMLPClassifier(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = RobertaModel(config)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, config.num_labels),
        )
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs[0][:, 0]  # CLS token
        logits = self.classifier(self.dropout(pooled_output))
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        return {"loss": loss, "logits": logits}

# Load model
from transformers import RobertaConfig
config = RobertaConfig.from_pretrained(base_model, num_labels=num_labels, id2label=id2label)
model = RobertaMLPClassifier.from_pretrained(base_model, config=config)

# PEFT: LoRA + LayerNorm + Bias
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias='all',
    target_modules=['query', 'value'],
    task_type="SEQ_CLS",
)
model = get_peft_model(model, peft_config)

# Verify trainable param count
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {count_trainable_params(model):,}")

# Evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds)}


Some weights of RobertaMLPClassifier were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.3.bias', 'classifier.3.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 595,716


In [ ]:

# Training config
training_args = TrainingArguments(
    output_dir="results",
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    report_to=None,
    save_total_limit=1,
    save_strategy="no",
    logging_dir="logs",
    disable_tqdm=False
)


trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

# Train
trainer.train()

# Evaluation function
import evaluate
def evaluate_model(inference_model, dataset, labelled=True, batch_size=8, data_collator=None):
    dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inference_model.to(device)
    inference_model.eval()

    all_predictions = []
    metric = evaluate.load("accuracy") if labelled else None

    for batch in tqdm(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = inference_model(**batch)
        preds = outputs["logits"].argmax(dim=-1)
        all_predictions.append(preds.cpu())
        if labelled:
            metric.add_batch(predictions=preds.cpu().numpy(), references=batch["labels"].cpu().numpy())

    all_predictions = torch.cat(all_predictions, dim=0)
    if labelled:
        return metric.compute(), all_predictions
    return all_predictions

# Final eval
print("Final Evaluation:")
_ = evaluate_model(model, eval_dataset, True, 8, data_collator)


<ipython-input-15-cc2f021403a1>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


OutOfMemoryError: CUDA out of memory. Tried to allocate 240.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 44.12 MiB is free. Process 4160 has 14.70 GiB memory in use. Of the allocated memory 14.44 GiB is allocated by PyTorch, and 131.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
#Load your unlabelled data
unlabelled_dataset = pd.read_pickle("test_unlabelled.pkl")
test_dataset = unlabelled_dataset.map(preprocess, batched=True, remove_columns=["text"])
df = unlabelled_dataset.to_pandas()
df.to_csv("test_unlabelled.csv", index=False)
# Run inference and save predictions
preds = evaluate_model(model, test_dataset, False, 8, data_collator)
df_output = pd.DataFrame({
    'ID': range(len(preds)),
    'Label': preds.numpy()  # or preds.tolist()
})


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

100%|██████████| 1000/1000 [01:46<00:00,  9.39it/s]


NameError: name 'output_dir' is not defined

In [ ]:
df_output.to_csv(os.path.join("results","inference_output.csv"), index=False)
print("Inference complete. Predictions saved to inference_output.csv")

Inference complete. Predictions saved to inference_output.csv
